In [ ]:
from snowflake.snowpark.functions import *
from snowflake.snowpark.context import get_active_session

In [ ]:
session = get_active_session()

# upload employees.csv --> EMPLOYEES table in TEST.PUBLIC db schema
df = session.table("exercise_db.public.employees")
df

In [ ]:
res = df.agg({"salary": "sum", "*": "count"})
res
res = df.agg(("salary", "sum"), ("*", "count"))
res
res = df.agg(sum("salary").alias("s"), count("*").alias("c"))
res

In [ ]:
res = df.group_by("department").agg({"salary": "sum", "*": "count"})
res
res = df.group_by("department").sum("salary")
res
res = df.group_by("department").function("sum")("salary")
res

In [ ]:
df.group_by("department").agg(("salary", "sum"), ("salary", "max"))

In [ ]:
# ~ drop_duplicates()
df.select("department").distinct()

In [ ]:
# employees w/ top salary per department (else w/ row_number!)
df.sort("salary").drop_duplicates("department")

In [ ]:
from snowflake.snowpark import GroupingSets

df.group_by("department","job").agg(("*","count"))

#gs = GroupingSets([col("department")], [col("job")])
#df.group_by_grouping_sets(gs).count()

In [ ]:
from snowflake.snowpark.functions import grouping

# this grouping returns a bitmast value 


 #0                             | grouped by both    |
 #1                             | job is grouped out |
 #2                             | department is out  |
 #3                             | grand total        |

df.group_by_grouping_sets(gs).agg([
        count("*").alias("count"),
        grouping("department", "job").alias("g")   
    ]).sort("g", "department", "job")

In [ ]:
df2 = df.rollup("department", "job").count()
df2

df2 = df.rollup("department", "job").agg([
        count("*").alias("count"),
        grouping("department", "job").alias("g")
    ]).sort("g", "department", "job")
df2

In [ ]:
df2 = df.cube("department", "job").count()
df2

df2 = df.cube("department", "job").agg([
        count("*").alias("count"),
        grouping("department", "job").alias("g")
    ]).sort("g", "department", "job")
df2

In [ ]:
df2 = df.select("department", "job", "salary"
    ).filter(col("job").isin(['CLERK', 'MANAGER', 'ACCOUNTANT']))
df2

# CROSSTAB
dfc = df2.crosstab("department", "job")
dfc

In [ ]:
df3 = df2.group_by("department", "job").sum("salary")
df3

# PIVOT
dfp = df2.pivot("job", ['CLERK', 'MANAGER', 'ACCOUNTANT']).sum("salary")
dfp

# UNPIVOT
dfu = dfp.unpivot("salary", "job", ["'CLERK'", "'MANAGER'", "'ACCOUNTANT'"])
dfu